# Index občanské vybavenosti

Reflektuje dostupnost a kapacitu klíčových služeb (škola, lékař, kulturní instituce atp.) v dané lokalitě. Je konstruován jako součet přítomnosti vybraných typů služeb a infrastruktury v obci (0 = nepřítomno, 1 = přítomno). Škála indexu se pohybuje od 0 do 17.

Vytvoření indexu:
- data o Místech poskytování zdravotnických služeb - přepočet jednotlivých typů zařízení na jejich počet v obci
- data z Registr ekonomických subjektů - přepočet jednotlivých typů subjektů na jejich počet v obci
- data PAQ research (2024) - doplnění složek občanské vybavenosti o dětské hřiště, sportoviště, centrum pro volný čas
- data MŠMT - přítomnost ZŠ a MŠ v obcích

In [5]:
import pandas as pd

#zdravotnická zařízení - praktik, pediatr, zubar, gynekolog, lékarna
ZZ_vybavenost = pd.read_csv("zdravotnicka_zarizeni_vybavenost.csv", sep=",", encoding="utf-8")

ZZ_vybavenost = ZZ_vybavenost.groupby(["KodObec", "ZZ_druh_nazev"])["ZZ_ID"].count().reset_index()
ZZ_vybavenost = ZZ_vybavenost.pivot_table(index="KodObec", columns="ZZ_druh_nazev", values="ZZ_ID", fill_value=0).reset_index()
ZZ_vybavenost.columns.name = None
ZZ_vybavenost = ZZ_vybavenost.drop(columns=["Centrum komplexní péče o děti","Poskytovatel amb. služeb (do 5 oborů)","Poskytovatel amb. služeb (nad 5 oborů)","Zařízení závodní preventivní péče"])

# pocty_obec.to_csv("ZZ_pocty_obec.csv", index=False, sep=",", encoding="utf-8") # počty jednotlivých zařízení na obec
# přítomnost daného zařízení v obci
# všechny sloupce kromě KodObec/IdOkrsek nahradit 0 nebo 1
sloupce = [c for c in ZZ_vybavenost.columns if c != "KodObec"]
ZZ_vybavenost[sloupce] = ZZ_vybavenost[sloupce].clip(upper=1).astype("Int64") 
ZZ_vybavenost["KodObec"] = ZZ_vybavenost["KodObec"].astype(str)

In [6]:
import pandas as pd

# hospoda/restaurace, obchod
#obec
RES_vybavenost = pd.read_csv("RES_pocty_obce.csv")
# všechny sloupce kromě KodObec/IdOkrsek nahradit 0 nebo 1
RES_vybavenost = RES_vybavenost.drop(columns=['pocet_jine', 'pocet_kadernictvi','pocet_vzdelavani','pocet_zdravotnictvi'])
sloupce = [c for c in RES_vybavenost.columns if c != "KodObec"]
RES_vybavenost[sloupce] = RES_vybavenost[sloupce].clip(upper=1).astype("Int64") 
RES_vybavenost["KodObec"] = RES_vybavenost["KodObec"].astype(str)

RES_vybavenost.head()


,KodObec,pocet_knihovny_muzea,pocet_maloobchod,pocet_sport_rekreace,pocet_stravovani,pocet_umeni,pocet_verejna_sprava
0,500011,0,1,1,1,1,1
1,500020,0,1,1,1,0,1
2,500046,0,1,1,1,1,1
3,500062,0,1,1,1,0,1
4,500071,0,1,1,1,1,1


In [7]:
import pandas as pd
# dětské hřiště/sportoviště/centrum pro volný čas

paq_vybavenost = pd.read_csv("dataPAQ_upravena/data_paq_joined.csv")

paq_vybavenost = paq_vybavenost.drop(columns=
        ['Nazev_obce', 'Kod_okresu', 'Nazev_okresu', 'Kod_kraje','Nazev_kraje', 
        'Prispevky_bydleni_avgMonth_2024',
       'Doplatky_bydleni_avgMonth_2024', 'Pridavky_deti_avgMonth_2024',
       'Prispevky_pece_avgMonth_2024', 'Prispevky_zivobyti_avgMonth_2024',
       'NEZAMESTNANOST', 'Dlouhodoba_nezamestnanost_2024',
       'Role_obce_na_predskolni_peci_2021', 'VEK_65AVIC', 'VEK_15_64', 'OBYVATELSTVO_POCET_2024',
       'POCET_NAROZENYCH_DETI', 'MUZI_SAMOZIVITELE', 'ZENY_SAMOZIVITELKY',
       'DETI_3_4_BEZ_DOCHAZKY_DO_MS', 'LIDE_EXEKUCE',
       'VEK_0_14','STREDISKO_VOLNY_CAS'])

paq_vybavenost = paq_vybavenost.rename(columns={"Kod_obce": "KodObec"})
paq_vybavenost["KodObec"] = paq_vybavenost['KodObec'].astype(str)
sloupce_obj = [c for c in paq_vybavenost.select_dtypes(include="str").columns if c != "KodObec" and c != "HUSTOTA_ZALIDNENI"]
paq_vybavenost[sloupce_obj] = paq_vybavenost[sloupce_obj].replace("data chybí", pd.NA)

paq_vybavenost["DETSKE_HRISTE"] = paq_vybavenost['DETSKE_HRISTE'].astype("Int64")
paq_vybavenost["PECOVATELAK"] = paq_vybavenost['PECOVATELAK'].astype("Int64")
paq_vybavenost["OBECNI_POLICIE"] = paq_vybavenost["OBECNI_POLICIE"].map({
    "Smlouva o policii": 1,
    "Soběstačný zřizovatel policie": 1,
    "Nemá policii": 0
}).fillna(0)
sloupce = [c for c in paq_vybavenost.columns if c != "KodObec" and c != "HUSTOTA_ZALIDNENI"]
paq_vybavenost[sloupce] = paq_vybavenost[sloupce].apply(pd.to_numeric, errors='coerce').clip(upper=1).astype("Int64")
paq_vybavenost["HUSTOTA_ZALIDNENI"] = pd.to_numeric(paq_vybavenost["HUSTOTA_ZALIDNENI"], errors='coerce')

In [8]:
# zš, mš

import pandas as pd
skoly_vybavenost = pd.read_csv("ZS_MS_SS_krouzky_obce.csv")

skoly_vybavenost = skoly_vybavenost.drop(columns=['Domov mládeže', 'Dům dětí a mládeže',
       'Jazyková škola s právem státní jazykové zkoušky', 'Konzervatoř','Plavecká škola', 'Stanice zájmových činností',
       'Středisko volného času', 'Střední škola', 'Vyšší odborná škola',
       'Základní umělecká škola',
       'Školní klub', 'Školní knihovna' ])

sloupce = [c for c in skoly_vybavenost.columns if c != "KodObec"]
skoly_vybavenost[sloupce] = skoly_vybavenost[sloupce].clip(upper=1).astype("Int64") 
skoly_vybavenost["KodObec"] = skoly_vybavenost['KodObec'].astype("Int64").astype(str)



In [9]:
# merge df pro vytvoření indexu vybavenosti

obcanska_vybavenost = paq_vybavenost.merge(ZZ_vybavenost, on="KodObec", how="left")
obcanska_vybavenost = obcanska_vybavenost.merge(RES_vybavenost, on="KodObec", how="left")
obcanska_vybavenost = obcanska_vybavenost.merge(skoly_vybavenost, on="KodObec", how="left")
obcanska_vybavenost = obcanska_vybavenost.fillna(0)
print(len(obcanska_vybavenost))

obcanska_vybavenost = obcanska_vybavenost.rename(columns={"Sam.ord.prakt.lékaře pro děti a dorost":"Pediatr",
                                                          "Samost. ordinace všeob. prakt. lékaře":"Praktik", 
                                                          "Samostatná ordinace PL - gynekologa":"Gynekolog", 
                                                          "Samostatná ordinace PL - stomatologa":"Stomatolog", 
                                                          "pocet_maloobchod":"Obchod",
                                                          "pocet_sport_rekreace":"Sport_rekreace",
                                                          "pocet_stravovani":"Stravovani",
                                                          "DETSKE_SKUPINY":"Detske_skupiny",
                                                          "DETSKE_HRISTE":"Detske_hriste",
                                                          "Mateřská škola":"Materska_skola",
                                                          "Základní škola": "Zakladni_skola",
                                                          "OBECNI_POLICIE": "Policie",
                                                          "pocet_verejna_sprava": "Verejna_sprava"
                                                          })



#obcanska_vybavenost.columns.to_list

6268


In [11]:
# index
obcanska_vybavenost["index_vybavenosti"] = (
    obcanska_vybavenost["Lékárna"] +
    obcanska_vybavenost["Pediatr"] +
    obcanska_vybavenost["Praktik"] +
    obcanska_vybavenost["Gynekolog"] +
    obcanska_vybavenost["Stomatolog"] +
    obcanska_vybavenost["Obchod"] +
    obcanska_vybavenost["Sport_rekreace"] +
    obcanska_vybavenost["Stravovani"] +
    obcanska_vybavenost["Detske_skupiny"] +
    obcanska_vybavenost["Detske_hriste"] +
    obcanska_vybavenost["Materska_skola"] +
    obcanska_vybavenost["Zakladni_skola"]+
    obcanska_vybavenost["Policie"]+
    obcanska_vybavenost["Verejna_sprava"]+
    obcanska_vybavenost["Školní družina"]+
    obcanska_vybavenost["pocet_knihovny_muzea"]+
    obcanska_vybavenost["pocet_umeni"]
)

# distribuce vybavevenosti
#obcanska_vybavenost["index_vybavenosti"].value_counts().sort_index()

obcanska_vybavenost = obcanska_vybavenost.drop(columns=['PECOVATELAK', 'HUSTOTA_ZALIDNENI', 'Detske_skupiny',
       'Detske_hriste', 'KULTURAK', 'KINO', 'KOSTEL', 'Policie', 'Lékárna',
       'Pediatr', 'Praktik', 'Gynekolog', 'Stomatolog', 'pocet_knihovny_muzea',
       'Obchod', 'Sport_rekreace', 'Stravovani', 'pocet_umeni',
       'Verejna_sprava', 'Materska_skola', 'Zakladni_skola', 'Školní družina'])
obcanska_vybavenost.to_csv("index_obcanska_vybavenost.csv", sep=",", index=False, encoding="utf-8")
